In [1]:
import glob
import os
import pandas as pd
import numpy as np
import re
import zipfile
from typing import List, Tuple

In [2]:
def is_header_row(row) -> bool:
    return any(
        str(cell).strip().lower() in ['nombre', 'posición', 'pos']
        for cell in row
    )

def is_section_title(row) -> bool:
    pattern = re.compile(r"(catchers|pitchers|infielders|outfielders)", re.IGNORECASE)
    return any(pattern.fullmatch(str(cell).strip()) for cell in row)

def extract_table_sections_from_excel(filepath: str, preview_rows: int = 5) -> List[pd.DataFrame]:
    cols = ['id','raw_name','position']
    df_raw = pd.read_excel(filepath, header=None)
    start_rows = []
    title_rows = []

    for i in range(len(df_raw)):
        row = df_raw.iloc[i]
        if is_header_row(row) or is_section_title(row):
            title = next((str(cell).strip().lower() for cell in row if str(cell).strip().lower() in ['catchers', 'pitchers', 'infielders', 'outfielders']), None)
            start_rows.append(i)
            title_rows.append(title)

    tables = []
    for idx, start in enumerate(start_rows):
        end = start_rows[idx + 1] if idx + 1 < len(start_rows) else None
        section = df_raw.iloc[start:end]
        section.columns = section.iloc[0]
        section = section[1:]  # remove header row
        section.dropna(axis=1,how='all',inplace=True)
        section.dropna(axis=0,how='all',inplace=True)
        if len(section.columns):
            section.columns = [cols[i] if i < 3 else 'other' for i in range(len(section.columns)) ]
            section = section[['id', 'raw_name', 'position']]
        section = section.reset_index(drop=True)
        section['title'] = title_rows[idx]
        tables.append(section.head(preview_rows))

    return tables

In [121]:
# import pdfplumber
import camelot

def extract_pdf_tables_with_titles(pdf_path: str) -> pd.DataFrame:
    df_temp = camelot.read_pdf(pdf_path, pages='all', flavor='lattice')[0].df
    if df_temp[1].isin(['PITCHERS', 'CUERPO TECNICO']).any():
        if df_temp[3].str.strip().isin(['P']).any():
            df_temp = df_temp[df_temp[3].str.strip() == 'P']
            df_temp = df_temp.drop(0,axis=1).loc[7:,2]
            return df_temp
        else:
            df_temp = df_temp.drop(0,axis=1).loc[7:,1]

        idx_pitcher = df_temp[df_temp=='PITCHERS'].index[0]
        idx_ct = df_temp[df_temp=='CUERPO TECNICO'].index[0]
        df_temp = df_temp.loc[idx_pitcher+1:idx_ct-1]
    else:
        df_temp = camelot.read_pdf(pdf_path, pages='all', flavor='lattice')[0].df
        df_temp = df_temp.drop(0,axis=1).loc[7:,[1,2]]
        df_temp = df_temp[df_temp[2].str.strip() == 'P']

    return df_temp

In [122]:
def extract_and_preview_documents(zip_path: str, extract_to: str = "temp_docs") -> dict:
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    content = {}
    for filename in os.listdir(extract_to):
        filepath = os.path.join(extract_to, filename)

        if filename.lower().endswith(('.xlsx', '.xls')):
            try:
                tables = extract_table_sections_from_excel(filepath)
                content[filename] = tables
            except Exception as e:
                content[filename] = [f"Error al leer Excel: {e}"]

        elif filename.lower().endswith('.pdf'):
            try:
                text_preview = extract_pdf_tables_with_titles(filepath)
                text_preview.dropna(inplace=True)
                content[filename] = [text_preview]
            except Exception as e:
                content[filename] = [f"Error al leer PDF: {e}"]

        else:
            content[filename] = ["Tipo de archivo no soportado"]
    return content

In [123]:
preview = extract_and_preview_documents("roosters.zip")

for filename, sections in preview.items():
    print(f"--- {filename} ---")
    for i, section in enumerate(sections):
        print(f"[Sección {i+1}]")
        print(section)
        print("\n")
    print("\n" + "-"*80 + "\n")

C:\Users\serch\anaconda3\envs\lmb\Lib\site-packages\camelot\utils.py:740: UserWarning:   (582.7, 585.1360000000001) does not lie in column range (36.54, 582.48)
  warnings.warn(


--- ROSTER CHI 10-JULIO-2025.xlsx ---
[Sección 1]
    id        raw_name position title
0  1.0  SERGIO BURRUEL        C  None
1  2.0    HANSEN LOPEZ        C  None
2  3.0      JOSE GODOY        C  None
3  4.0  RUSBER ESTRADA        C  None
4  5.0  JESUS CASTILLO       IF  None



--------------------------------------------------------------------------------

--- ROSTER DUR 10-JULIO-2025.pdf ---
[Sección 1]
22           LUIS RIJO
23    JONATHAN PARTIDA
24     TIAGO  DA SILVA
25       JORGE SAUCEDA
26        JORGE GARCIA
27         EDWYN VALLE
28      RJ MARTINEZ JR
29        HECTOR PEREZ
30      JORGE BAUTISTA
31        TANNER TULLY
32       MANUEL CHAVEZ
33      EMILKER GUZMAN
34       FELIPE ACOSTA
35       IRVING MEDINA
36          REGI GRACE
37           JOSE CRUZ
38     KEYVIUS SAMPSON
Name: 1, dtype: object



--------------------------------------------------------------------------------

--- ROSTER LAG 10-JULIO-2025.xlsx ---
[Sección 1]
Empty DataFrame
Columns: [title]
Index:

## Debug pdf extract

In [20]:
extract_to = 'temp_docs'
content = {}
for filename in os.listdir(extract_to):
    filepath = os.path.join(extract_to, filename)
    print(filepath)

temp_docs\ROSTER CHI 10-JULIO-2025.xlsx
temp_docs\ROSTER DUR 10-JULIO-2025.pdf
temp_docs\ROSTER LAG 10-JULIO-2025.xlsx
temp_docs\ROSTER MTY 10-JULIO-2025.pdf
temp_docs\ROSTER MVA 10-JULIO-2025.xlsx
temp_docs\ROSTER PUE 10-JULIO-2025.xlsx
temp_docs\ROSTER VER 10-JULIO-2025.pdf


In [285]:
p0.

580

In [104]:
df_temp = camelot.read_pdf('temp_docs\ROSTER VER 10-JULIO-2025.pdf', pages='all', flavor='lattice')[0].df  # o 'stream'
df_temp.head(40)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,,,,,,,,,,,,,
1,EL AGUILA DE VERACRUZ\n10 DE JULIO,,,,,,,,,,,,
2,NOMBRE\nEDAD\nT/B\nPESO\nESTATURA\nFECHA DE NA...,,,,,,,,,,,,
3,,CATCHER,,,,,,,,,,,
4,,1\nJUAN BERNABE URIARTE SOTO\nC\n72\n27\nD/D\n...,,,,,,,,,,,
5,,2\nSAUL GARZA\nC\n6\n27\nD/D\n99\n1.85\n4/9/19...,,,,,,,,,,,
6,,3\nMIGUEL ARTURO OJEDA\nC\n2\n25\nD/D\n90\n1.8...,,,,,,,,,,,
7,,INFIELD,,,,,,,,,,,
8,,4,DAVID ALEXANDER RODRIGUEZ RINCONES,INF,9,E,28,D/D\n97,,1.85,2/25/1996,"BARCELONA , VENEZUELA",
9,,5,ERNEST NIKO VASQUEZ,INF,3,,36,D/D\n77,,1.83,26/02/1989,"OXNARD, CALIFORNIA",


In [119]:
def extract_pdf_tables_with_titles(pdf_path: str) -> pd.DataFrame:
    df_temp = camelot.read_pdf(pdf_path, pages='all', flavor='lattice')[0].df
    if df_temp[1].isin(['PITCHERS', 'CUERPO TECNICO']).any():
        if df_temp[3].str.strip().isin(['P']).any():
            df_temp = df_temp[df_temp[3].str.strip() == 'P']
            df_temp = df_temp.drop(0,axis=1).loc[7:,2]
            return df_temp
        else:
            df_temp = df_temp.drop(0,axis=1).loc[7:,1]

        idx_pitcher = df_temp[df_temp=='PITCHERS'].index[0]
        idx_ct = df_temp[df_temp=='CUERPO TECNICO'].index[0]
        df_temp = df_temp.loc[idx_pitcher+1:idx_ct-1]
    else:
        df_temp = camelot.read_pdf(pdf_path, pages='all', flavor='lattice')[0].df
        df_temp = df_temp.drop(0,axis=1).loc[7:,[1,2]]
        df_temp = df_temp[df_temp[2].str.strip() == 'P']

    return df_temp

In [120]:
extract_pdf_tables_with_titles('temp_docs\ROSTER VER 10-JULIO-2025.pdf')

23    NORMAN SALVADOR ELENES QUINTERO
24                 CONNOR RODD CURLIS
25           DINELSON LAMET HERNANDEZ
26       ESMIL ANTONIO ROGERS VASQUEZ
27                       JONATHAN ARO
28                       ORLANDO LARA
29      LUIS MIGUEL ROMERO MASFARROLL
30              NICK JORDAN GARDEWINE
31        LUIS ERNESTO MARQUEZ FLORES
32             ALBERTO LEYVA PEÑUÑURI
33                MARIO JIMENEZ MARIN
34        JESUS FABIAN ANGUAMEA SERNA
35             VIDAL VICENTE NUÑO III
36            VICTOR BUELNA  ESPINOZA
37       ALEJANDRO TRUJILLO CERVANTES
Name: 2, dtype: object